# Figure 5 — Dataset Distribution by Demographic Thresholds

Three-panel stacked bar chart showing the sample composition of the
training and test splits across Age, BMI, and Sex thresholds.

- **Bar fill (%)** — fraction of *samples* (segments) in each category
- **Annotation to the right** — number of unique *subjects* in each category

Values are hardcoded from the dataset analysis (static dataset properties).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import FIGURES_PAPER

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

FIG_OUT = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory : {FIG_OUT.resolve()}")
print(f"Full-page  : {W_FULL:.2f} x {W_FULL*ASPECT:.2f} in  ->"
      f"  {round(W_FULL*DPI)} x {round(W_FULL*ASPECT*DPI)} px @ {DPI} dpi")

## 1 · Data
Each entry: `(sample_pct, n_subjects)` per category, ordered bottom-to-top.

In [ ]:
# ── Dataset composition (from dataset analysis) ───────────────────────────────
# Each tuple: (sample_percentage, n_unique_subjects)
# Colors are all dark enough to be readable with white text labels.

PANELS = [
    {
        "title":       "Age Distributions by Thresholds",
        "panel_label": "A)",
        "categories":  ["<40", "40–60", "≥60"],
        "train": [(10.5, 136), (34.6, 448), (54.8, 709)],
        "test":  [(13.9,  20), (33.3,  48), (52.8,  76)],
        "colors": ["#3498DB", "#1F618D", "#154360"],   # light → mid → dark blue
    },
    {
        "title":       "BMI Distributions by Thresholds",
        "panel_label": "B)",
        "categories":  ["<25", "≥25"],
        "train": [(74.8, 967), (25.2, 326)],
        "test":  [(79.2, 114), (20.8,  30)],
        "colors": ["#148F77", "#0B5E42"],              # teal → dark green
    },
    {
        "title":       "Sex Distributions by Thresholds",
        "panel_label": "C)",
        "categories":  ["Male", "Female"],
        "train": [(57.7, 746), (42.3, 547)],
        "test":  [(59.7,  86), (40.3,  58)],
        "colors": ["#2980B9", "#B03A2E"],              # blue / dark crimson
    },
]

## 2 · Plot

In [ ]:
BAR_W          = 0.50
LABEL_THRESH   = 7.0    # min % to print label inside a section
SUBJ_X_OFFSET  = 0.04   # horizontal gap between bar edge and subject count


def draw_panel(ax, panel: dict, width_in: float):
    is_small = width_in < 5
    pct_fs   = 6.5 if is_small else 9.0
    subj_fs  = 7.0 if is_small else 9.0   # larger + bold for subject counts
    tick_fs  = 6   if is_small else 8
    title_fs = 6.5 if is_small else 10
    leg_fs   = 5.5 if is_small else 7.5

    for col_idx, (split_name, split_data) in enumerate(
        [("Train", panel["train"]), ("Test", panel["test"])]   # ① no % sign
    ):
        bottom = 0.0
        for cat_idx, (pct, n_subj) in enumerate(split_data):
            color = panel["colors"][cat_idx]
            ax.bar(
                col_idx, pct, BAR_W,
                bottom=bottom,
                color=color,
                edgecolor="black", linewidth=0.5,
                zorder=3,
            )
            bar_cy = bottom + pct / 2

            # ④ No bold on % label  ⑤ All white text
            if pct >= LABEL_THRESH:
                ax.text(
                    col_idx, bar_cy, f"{pct:.1f}%",
                    ha="center", va="center",
                    fontsize=pct_fs, fontfamily=FONT_FAMILY,
                    color="white",
                    fontweight="bold",
                    zorder=4,
                )

            # ③ Subject count — larger font + bold
            ax.text(
                col_idx + BAR_W / 2 + SUBJ_X_OFFSET, bar_cy,
                str(n_subj),
                ha="left", va="center",
                fontsize=subj_fs, fontfamily=FONT_FAMILY,
                color="#222222",
                fontweight="bold",
                zorder=4,
            )
            bottom += pct

    # Axes
    ax.set_xlim(-0.45, 1.9)
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Train", "Test"], fontsize=tick_fs, fontfamily=FONT_FAMILY)  # ① no %
    ax.set_title(panel["title"], fontsize=title_fs, fontfamily=FONT_FAMILY, pad=4)

    apply_base_style(ax, grid_axis="y")
    ax.tick_params(axis="both", labelsize=tick_fs)

    # ② Panel letter below x-axis tick labels (as xlabel)
    ax.set_xlabel(
        panel["panel_label"],
        fontsize=tick_fs + 2, fontfamily=FONT_FAMILY,
        fontweight="bold", labelpad=6,
    )

    # Legend below the panel letter
    legend_patches = [
        Patch(facecolor=panel["colors"][i], edgecolor="black", linewidth=0.5,
              label=panel["categories"][i])
        for i in range(len(panel["categories"]))
    ]
    ax.legend(
        handles=legend_patches,
        loc="lower center", ncol=len(panel["categories"]),
        fontsize=leg_fs,
        frameon=True, framealpha=0.9, edgecolor="#CCCCCC",
        bbox_to_anchor=(0.5, -0.28),
    )


def make_fig5(width_in: float):
    height_in = width_in * ASPECT
    fig, axes = plt.subplots(
        1, 3,
        figsize=(width_in, height_in),
        dpi=DPI,
        layout="constrained",
    )
    for ax, panel in zip(axes, PANELS):
        draw_panel(ax, panel, width_in)
        ax.set_ylabel("")

    axes[0].set_ylabel("Sample Proportion", fontsize=8 if width_in < 5 else 10,
                       fontfamily=FONT_FAMILY)
    return fig


fig_full   = make_fig5(W_FULL)
fig_single = make_fig5(W_SINGLE)

save_fig(fig_full,   "Fig5_Dataset_Demographics_full",   FIG_OUT)
save_fig(fig_single, "Fig5_Dataset_Demographics_single", FIG_OUT)

print(f"Full-page  : {round(W_FULL*DPI)} x {round(W_FULL*ASPECT*DPI)} px")
print(f"Single-col : {round(W_SINGLE*DPI)} x {round(W_SINGLE*ASPECT*DPI)} px")
plt.show()

## 3 · Verify pixel counts

In [ ]:
from PIL import Image

for fname, req_w in [
    ("Fig5_Dataset_Demographics_full.png",   MIN_PX_FULL),
    ("Fig5_Dataset_Demographics_single.png", MIN_PX_SINGLE),
]:
    with Image.open(FIG_OUT / fname) as im:
        w, h = im.size
    ok = "\u2705" if w >= req_w else "\u274c"
    print(f"{ok} {fname}: {w} x {h} px  (min required: {req_w} px wide)")